🟢 Débutant | ⏱ 4 à 6 h | 🔑 projet : pandas, SQLite, objets, matplotlib, fichiers

# Module 13 — Projet final : le bulletin complet

> 🎬 **Regarde d'abord la vidéo du module** (scénario `python-debutant-13`, à venir).

## L'histoire

Dernier lundi de ton stage chez MétéoLocale. Inès t'attend avec un carton de croissants :

> « Tu as fait chaque brique séparément. Aujourd'hui, tu les assembles. Je te donne le fichier **brut** que nous
> envoient les stations, tel quel, avec ses défauts. Je veux, à la fin, sans rien retoucher à la main :
> une base de données propre, un tableau par ville, une prévision, un graphique, et le bulletin en texte et en JSON.
> Si demain on me donne le fichier d'octobre, je relance, et tout se refait tout seul. »

C'est ce qu'on appelle un **pipeline** (un « tuyau ») : une suite d'étapes où la sortie de l'une est l'entrée de la
suivante, comme une chaîne de montage.

```
meteo_brut.csv ─▶ nettoyage ─▶ base SQLite ─▶ statistiques ─▶ prévision ─▶ graphique ─▶ bulletin .txt / .json
   (module 08)    (module 10)   (module 12)     (module 10)     (module 11)   (module 10)    (modules 08, 09)
```

## Ce qui est nouveau (un peu)

Presque tout a déjà été vu ; chaque étape te dit **dans quel module** retrouver le geste. Trois petites choses sont
nouvelles, et expliquées sur place quand elles arrivent :
- `tableau.copy()` (étape 3) : un double indépendant d'un DataFrame, pour ne jamais abîmer l'original ;
- un **dictionnaire dans un dictionnaire** (étape 14) : une fiche dont une rubrique contient d'autres fiches ;
- les conversions `float(...)` et `int(...)` avant `json.dump` (étape 14) : `json` ne connaît que les nombres Python
  de base, pas ceux de pandas.

## Comment travailler

1. Lis l'**objectif** et les **étapes**.
2. Remplace chaque `...` par ton code ; garde les noms de variables déjà écrits, la vérification en a besoin.
3. Exécute ta cellule (`Shift + Entrée`) et compare au **résultat attendu**.
4. Exécute la cellule **Vérification** : `✅ OK` = réussi ; sinon lis le message, il te dit quoi corriger.
5. Bloqué plus de 5 minutes ? Ouvre l'indice 💡, puis relis le module indiqué.

**Lire un échec.** Si la vérification échoue, la dernière ligne commence par `AssertionError:`. Ça veut dire « la vérification a échoué » (ce n'est pas Python qui est cassé). Le message **en français** après les deux-points te dit quoi corriger ; les lignes en anglais au-dessus, tu peux les ignorer. Les caractères spéciaux (`°`, `—`, `«`, `î`…) : copie-colle-les depuis l'énoncé.

⚠️ Ici, **les étapes s'enchaînent** : chaque étape utilise les variables des étapes précédentes. Exécute-les
**dans l'ordre**. Si tu redémarres le kernel, réexécute tout depuis le début, cellule par cellule (ou d'un coup : Jupyter, menu *Run → Run All Cells* ;
VS Code, bouton *Run All* en haut du notebook).

Tout ce que le projet écrit va dans `sorties/`. Exécute la cellule de préparation :

In [ ]:
import json
import sqlite3
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

Path("sorties").mkdir(exist_ok=True)
print("Prêt : pandas", pd.__version__)

## Partie 1 — Charger et diagnostiquer

### Étape 1 — Charger le fichier brut

**Objectif** : Charger `meteo_brut.csv` dans un DataFrame.

**À faire** :
1. Charge `../donnees/meteo_brut.csv` dans `brut` (attention au séparateur `;`).
2. Range sa taille dans `taille_brut`.

**Résultat attendu** :
```
(185, 6)
```

<details><summary>💡 Indice 1 : la méthode</summary>

Charger un CSV avec un séparateur `;` (argument nommé `sep`), puis lire sa taille avec l'attribut `shape`.

📖 Module 10, cours, section 3.

</details>

<details><summary>🔑 Indice 2 : la ligne à écrire (si l'indice 1 ne suffit pas)</summary>

`pd.read_csv("../donnees/meteo_brut.csv", sep=";")` puis `brut.shape`.

</details>

In [ ]:
brut = pd.read_csv(...)
taille_brut = ...
print(taille_brut)
brut.head()

In [ ]:
# Vérification
assert "taille_brut" in dir() and taille_brut is not ..., "Exécute d'abord ta cellule."
assert taille_brut != (185, 1), "Une seule colonne : précise sep=\";\"."
assert taille_brut == (185, 6), f"taille_brut doit valoir (185, 6), tu as {taille_brut}."
print("✅ OK")

### Étape 2 — Le diagnostic

**Objectif** : Avant de réparer, on mesure les dégâts. Inès veut un **dictionnaire** `diagnostic` qui les compte.

**À faire** :
1. `"villes_differentes"` : le nombre de noms de villes différents (`set` + `len`). Il devrait y en avoir 6…
2. `"pluie_vide"` et `"vent_vide"` : le nombre de cases vides de `pluie_mm` et de `vent_kmh` (filtre `.isna()` + `len`).
3. `"doublons"` : le nombre de lignes en double (`len(brut) - len(brut.drop_duplicates())`).

**Résultat attendu** :
```
{'villes_differentes': 12, 'pluie_vide': 8, 'vent_vide': 6, 'doublons': 5}
```

<details><summary>💡 Indice 1 : la méthode</summary>

Même forme que la ligne `"pluie_vide"`, sur l'autre colonne. Pour les doublons, la formule est dans l'énoncé.

📖 Module 10, cours, sections 9.3 et 9.4 ; module 06 (dictionnaires).

</details>

<details><summary>🔑 Indice 2 : la ligne à écrire (si l'indice 1 ne suffit pas)</summary>

Même forme que `"pluie_vide"`, sur la colonne `vent_kmh`. Pour les doublons, la formule est dans l'énoncé.

</details>

In [ ]:
diagnostic = {
    "villes_differentes": len(set(brut["ville"])),
    "pluie_vide": len(brut[brut["pluie_mm"].isna()]),
    "vent_vide": ...,
    "doublons": ...,
}
print(diagnostic)

In [ ]:
# Vérification
assert "diagnostic" in dir(), "Exécute d'abord ta cellule."
assert diagnostic["villes_differentes"] == 12, "Il doit y avoir 12 noms différents (6 villes, écrites de deux façons)."
assert diagnostic["vent_vide"] == 6, f"vent_vide doit valoir 6, tu as {diagnostic['vent_vide']} : len(brut[brut[\"vent_kmh\"].isna()])."
assert diagnostic["doublons"] == 5, f"doublons doit valoir 5, tu as {diagnostic['doublons']}."
print("✅ OK")

## Partie 2 — Nettoyer, avec des fonctions réutilisables

Inès veut pouvoir relancer le nettoyage sur le fichier d'octobre : on range chaque réparation dans une **fonction**
(module 07) qui reçoit un DataFrame et renvoie le DataFrame réparé. On commence par faire une **copie** de travail
avec `brut.copy()` : une méthode de DataFrame qui renvoie un double indépendant, pour ne jamais abîmer l'original.

### Étape 3 — Réparer les villes

**Objectif** : Écrire `nettoyer_villes(tableau)` : enlève les espaces autour des villes et corrige les minuscules.

**À faire** :
1. Dans la fonction, remplace la colonne `ville` par `tableau["ville"].str.strip().replace(corrections)`.
2. La fonction renvoie `tableau`.
3. Applique-la à une copie de `brut` : `travail = nettoyer_villes(brut.copy())`.

**Résultat attendu** :
```
['Albi', 'Montpellier', 'Nîmes', 'Perpignan', 'Tarbes', 'Toulouse']
```

<details><summary>💡 Indice 1 : la méthode</summary>

La réparation des villes du module 10 : enlever les espaces avec la méthode de texte de la colonne, puis remplacer les valeurs entières avec le dictionnaire `corrections`. La fonction **renvoie** le tableau.

📖 Module 10, cours, section 9.1 ; module 07 (fonctions).

</details>

<details><summary>🔑 Indice 2 : la ligne à écrire (si l'indice 1 ne suffit pas)</summary>

La ligne du module 10 : `tableau["ville"].str.strip().replace(corrections)`.

</details>

In [ ]:
corrections = {
    "toulouse": "Toulouse", "montpellier": "Montpellier", "nîmes": "Nîmes",
    "perpignan": "Perpignan", "albi": "Albi", "tarbes": "Tarbes",
}


def nettoyer_villes(tableau):
    tableau["ville"] = ...
    return tableau


travail = nettoyer_villes(brut.copy())
print(sorted(set(travail["ville"])))

In [ ]:
# Vérification
assert "nettoyer_villes" in dir() and "travail" in dir(), "Exécute d'abord ta cellule."
essai = nettoyer_villes(pd.DataFrame([{"ville": "  albi "}, {"ville": "Tarbes"}]))
assert list(essai["ville"]) == ["Albi", "Tarbes"], f"nettoyer_villes doit transformer '  albi ' en 'Albi' : tu obtiens {list(essai['ville'])}. Fais .str.strip() AVANT .replace(corrections)."
assert sorted(set(travail["ville"])) == ["Albi", "Montpellier", "Nîmes", "Perpignan", "Tarbes", "Toulouse"], "travail doit contenir exactement les 6 villes bien écrites."
assert len(set(brut["ville"])) == 12, "brut a été modifié : applique la fonction à brut.copy(), pas à brut."
print("✅ OK")

### Étape 4 — Réparer les températures

**Objectif** : Écrire `convertir_temperatures(tableau)` : remplace la virgule décimale de `temp_max` par un point, puis convertit en nombres.

**À faire** :
1. Dans la fonction : `.str.replace(",", ".")` puis `pd.to_numeric(...)`, et `return tableau`.
2. Applique-la à `travail`.

**Résultat attendu** :
```
float64
```

<details><summary>💡 Indice 1 : la méthode</summary>

Deux temps du module 10 : remplacer la virgule **à l'intérieur** du texte, puis convertir la colonne en nombres. Si tu relances la cellule deux fois, `.str` plante (la colonne est déjà en nombres) : relance depuis l'étape 3.

📖 Module 10, cours, section 9.2.

</details>

<details><summary>🔑 Indice 2 : la ligne à écrire (si l'indice 1 ne suffit pas)</summary>

`pd.to_numeric(tableau["temp_max"].str.replace(",", "."))`. Si tu relances la cellule deux fois, `.str` plante (la colonne est déjà en nombres) : relance depuis l'étape 3.

</details>

In [ ]:
def convertir_temperatures(tableau):
    tableau["temp_max"] = ...
    return tableau


travail = convertir_temperatures(travail)
print(travail["temp_max"].dtype)

In [ ]:
# Vérification
assert "convertir_temperatures" in dir(), "Exécute d'abord ta cellule."
assert "travail" in dir(), "La variable travail n'existe pas : termine d'abord l'étape 3 (et l'étape 1)."
assert travail["temp_max"].dtype == "float64", f"temp_max doit être en float64, elle est en {travail['temp_max'].dtype}. Si l'erreur est 'Unable to parse string', remplace d'abord la virgule. Si l'erreur parle de '.str', relance depuis l'étape 3 (la colonne a déjà été convertie)."
assert travail["temp_max"].max() == 29.7, "Le maximum de temp_max doit valoir 29.7."
print("✅ OK")

### Étape 5 — Doublons et cases vides

**Objectif** : Enlever les doublons, puis les lignes incomplètes, et compter ce qu'on a perdu.

**À faire** :
1. Range dans `propre` : `travail` sans doublons, puis sans lignes à case vide (`drop_duplicates()` puis `dropna()`).
2. Range dans `nb_supprimees` le nombre de lignes enlevées depuis `brut`.

**Résultat attendu** :
```
166 19
```

<details><summary>💡 Indice 1 : la méthode</summary>

Deux méthodes enchaînées sur `travail`, de gauche à droite : d'abord les doublons, puis les lignes à case vide.

📖 Module 10, cours, sections 9.3 et 9.4.

</details>

<details><summary>🔑 Indice 2 : la ligne à écrire (si l'indice 1 ne suffit pas)</summary>

On enchaîne deux méthodes : `travail.drop_duplicates().dropna()`.

</details>

In [ ]:
propre = ...
nb_supprimees = len(brut) - len(propre)
print(len(propre), nb_supprimees)

In [ ]:
# Vérification
assert "propre" in dir() and propre is not ..., "Exécute d'abord ta cellule."
assert len(propre) != 180, "180 lignes : tu as enlevé les doublons mais pas les cases vides (dropna())."
assert len(propre) == 166, f"propre doit contenir 166 lignes, il en contient {len(propre)}."
print("✅ OK")

### Étape 6 — Le contrôle qualité

**Objectif** : Avant de ranger en base, on vérifie que les données sont plausibles. Range dans `controle_ok` un booléen : `True` si **toutes** les conditions sont vraies.

**À faire** :
1. Exactement 6 villes.
2. Aucune `temp_max` hors de l'intervalle −10 à 45 °C (un filtre avec `|` qui doit donner 0 ligne).
3. Aucune `pluie_mm` négative.
4. Combine les trois avec `and` (ce sont des booléens Python simples, pas des colonnes : ici, `and` est correct).

**Résultat attendu** :
```
True
```

<details><summary>💡 Indice 1 : la méthode</summary>

Même forme que `temperatures_ok` : un filtre qui doit donner 0 ligne. Puis les trois booléens reliés par `and` (module 03).

📖 Module 10, cours, section 6 ; module 03 (`and`).

</details>

<details><summary>🔑 Indice 2 : la ligne à écrire (si l'indice 1 ne suffit pas)</summary>

`pluies_ok = len(propre[propre["pluie_mm"] < 0]) == 0` ; `controle_ok = six_villes and temperatures_ok and pluies_ok`.

</details>

In [ ]:
six_villes = len(set(propre["ville"])) == 6
temperatures_ok = len(propre[(propre["temp_max"] < -10) | (propre["temp_max"] > 45)]) == 0
pluies_ok = ...
controle_ok = ...
print(six_villes, temperatures_ok, pluies_ok, controle_ok)

In [ ]:
# Vérification
assert "controle_ok" in dir() and controle_ok is not ..., "Exécute d'abord ta cellule."
assert pluies_ok is True, f"pluies_ok doit valoir True, tu as {pluies_ok} : len(propre[propre[\"pluie_mm\"] < 0]) == 0."
assert controle_ok is True, f"controle_ok doit valoir True, tu as {controle_ok} : six_villes and temperatures_ok and pluies_ok."
print("✅ OK")

## Partie 3 — Ranger dans la base

### Étape 7 — La base `meteolocale_final.db`

**Objectif** : Créer la base, à neuf, et y ranger les relevés propres avec une requête paramétrée.

**À faire** :
1. Ouvre `sorties/meteolocale_final.db` dans `connexion`.
2. Supprime la table `releves` si elle existe, puis crée-la avec les six colonnes et leurs types.
3. Insère chaque relevé de `propre` avec une requête paramétrée, puis valide.
4. Remarque : le vent a été lu en nombres à virgule (`41.0`, à cause des cases vides). Pas besoin de le convertir : une colonne `INTEGER` de SQLite range `41.0` comme l'entier `41`.

**Résultat attendu** :
```
166
```

<details><summary>💡 Indice 1 : la méthode</summary>

Les gestes du module 12 : `DROP TABLE IF EXISTS`, `CREATE TABLE` avec les types `TEXT`, `REAL`, `INTEGER`, une boucle d'`INSERT` à six `?` sur `propre.to_dict("records")`, puis `commit`.

📖 Module 12, cours, sections 3 et 4.

</details>

<details><summary>🔑 Indice 2 : la ligne à écrire (si l'indice 1 ne suffit pas)</summary>

```python
connexion.execute("DROP TABLE IF EXISTS releves")
connexion.execute("CREATE TABLE releves (date TEXT, ville TEXT, temp_min REAL, temp_max REAL, pluie_mm REAL, vent_kmh INTEGER)")
for f in propre.to_dict("records"):
    connexion.execute("INSERT INTO releves VALUES (?, ?, ?, ?, ?, ?)",
                      (f["date"], f["ville"], f["temp_min"], f["temp_max"], f["pluie_mm"], f["vent_kmh"]))
connexion.commit()
```

</details>

In [ ]:
connexion = sqlite3.connect("sorties/meteolocale_final.db")
# 1. Supprime la table releves si elle existe.
# 2. Crée la table releves (six colonnes, avec leurs types SQL).
# 3. Pour chaque fiche de propre (en liste de dictionnaires), insère une ligne avec six ?.
# 4. Valide.
...

print(len(connexion.execute("SELECT * FROM releves").fetchall()))

In [ ]:
# Vérification
v = sqlite3.connect("sorties/meteolocale_final.db")
tables = [t[0] for t in v.execute("SELECT name FROM sqlite_master WHERE type = 'table'").fetchall()]
assert "releves" in tables, "La table releves n'existe pas dans sorties/meteolocale_final.db : CREATE TABLE, puis commit."
n = v.execute("SELECT COUNT(*) FROM releves").fetchall()[0][0]
colonnes = [c[1] for c in v.execute("PRAGMA table_info(releves)").fetchall()]
v.close()
assert colonnes == ["date", "ville", "temp_min", "temp_max", "pluie_mm", "vent_kmh"], f"Colonnes attendues : date, ville, temp_min, temp_max, pluie_mm, vent_kmh. Tu as {colonnes}."
assert n == 166, f"La base doit contenir 166 relevés validés, elle en contient {n} : as-tu fait commit ?"
print("✅ OK")

### Étape 8 — Relire depuis la base

**Objectif** : Désormais, la **source de vérité**, c'est la base. On relit tout avec pandas.

**À faire** :
1. Range dans `historique` toute la table, avec `pd.read_sql`.
2. Range dans `nb_villes_base` le nombre de villes différentes.

**Résultat attendu** :
```
(166, 6) 6
```

<details><summary>💡 Indice 1 : la méthode</summary>

Lire toute une table dans un DataFrame : `pd.read_sql` avec une requête `SELECT *` et la connexion. Les villes différentes : `set` puis `len`.

📖 Module 12, cours, section 8.

</details>

<details><summary>🔑 Indice 2 : la ligne à écrire (si l'indice 1 ne suffit pas)</summary>

`pd.read_sql("SELECT * FROM releves", connexion)`.

</details>

In [ ]:
historique = pd.read_sql(..., connexion)
nb_villes_base = ...
print(historique.shape, nb_villes_base)

In [ ]:
# Vérification
assert "historique" in dir() and historique is not ..., "Exécute d'abord ta cellule."
assert historique.shape == (166, 6), f"historique doit faire (166, 6), il fait {historique.shape}."
assert nb_villes_base == 6, f"nb_villes_base doit valoir 6, tu as {nb_villes_base}."
print("✅ OK")

## Partie 4 — Statistiques et records

### Étape 9 — Le tableau par ville

**Objectif** : Le cœur du bulletin : une ligne par ville, de la plus chaude à la moins chaude.

**À faire** :
1. `tableau` : moyennes de `temp_min` et `temp_max` par ville, arrondies à 1 chiffre.
2. Ajoute `pluie_totale` (somme de `pluie_mm`, arrondie à 1 chiffre).
3. Ajoute `jours_pluie` : le nombre de jours où `pluie_mm > 0` (filtre, puis `groupby` + `.count()`).
4. Trie par `temp_max` décroissante.

**Résultat attendu** :
```
             temp_min  temp_max  pluie_totale  jours_pluie
ville                                                     
Nîmes            13.0      26.8         123.7           15
Perpignan        15.0      25.3         135.5           20
Montpellier      13.7      25.2         112.2           14
Toulouse         11.8      24.0         107.8           15
Albi             11.4      22.9         100.7           17
Tarbes            8.9      20.5          60.1           11
```

<details><summary>💡 Indice 1 : la méthode</summary>

Le tableau par ville du module 10, avec deux colonnes ajoutées : une somme, et un **comptage** sur les relevés pluvieux (filtre, puis la méthode qui compte). Puis un tri décroissant **rangé** dans `tableau`.

📖 Module 10, cours, sections 7 et 8.

</details>

<details><summary>🔑 Indice 2 : la ligne à écrire (si l'indice 1 ne suffit pas)</summary>

```python
tableau["pluie_totale"] = historique.groupby("ville")["pluie_mm"].sum().round(1)
pluvieux = historique[historique["pluie_mm"] > 0]
tableau["jours_pluie"] = pluvieux.groupby("ville")["pluie_mm"].count()
tableau = tableau.sort_values("temp_max", ascending=False)
```

</details>

In [ ]:
tableau = historique.groupby("ville")[["temp_min", "temp_max"]].mean().round(1)
# 1. Ajoute la colonne pluie_totale.
# 2. Garde les relevés pluvieux, puis ajoute la colonne jours_pluie (un comptage par ville).
# 3. Trie le tableau du plus chaud au moins chaud, et RANGE le résultat dans tableau.
...
tableau

In [ ]:
# Vérification
assert "tableau" in dir() and tableau is not ..., "Exécute d'abord ta cellule."
assert list(tableau.columns) == ["temp_min", "temp_max", "pluie_totale", "jours_pluie"], f"Colonnes attendues : temp_min, temp_max, pluie_totale, jours_pluie. Tu as {list(tableau.columns)}."
assert list(tableau.index)[0] == "Nîmes", f"La première ville doit être Nîmes (la plus chaude) : trie avec ascending=False."
assert tableau["pluie_totale"]["Toulouse"] == 107.8, f"La pluie totale de Toulouse doit valoir 107.8, tu as {tableau['pluie_totale']['Toulouse']}."
assert tableau["jours_pluie"]["Albi"] == 17, f"Albi doit avoir 17 jours de pluie, tu as {tableau['jours_pluie']['Albi']}."
print("✅ OK")

### Étape 10 — Les records, en SQL

**Objectif** : Trouver deux records directement dans la base, avec `ORDER BY`.

**À faire** :
1. `record_pluie` : le tuple `(ville, date, pluie_mm)` de la journée la plus pluvieuse (tri décroissant, première ligne).
2. `record_froid` : le tuple `(ville, date, temp_min)` de la nuit la plus froide (tri croissant).
3. `ville_chaude` : la première ville de `tableau`.

**Résultat attendu** :
```
('Nîmes', '2026-09-11', 22.1)
('Tarbes', '2026-09-29', 4.9)
Nîmes
```

<details><summary>💡 Indice 1 : la méthode</summary>

Un `SELECT … ORDER BY` décroissant, puis la première ligne de `fetchall` ; pareil en ordre croissant pour la nuit la plus froide. La ville la plus chaude : la première étiquette de `tableau`.

📖 Module 12, cours, section 5 ; module 10, cours, section 8.

</details>

<details><summary>🔑 Indice 2 : la ligne à écrire (si l'indice 1 ne suffit pas)</summary>

`ORDER BY pluie_mm DESC` puis `[0]` ; même chose avec `ORDER BY temp_min`. Puis `list(tableau.index)[0]`.

</details>

In [ ]:
record_pluie = connexion.execute("SELECT ville, date, pluie_mm FROM releves ORDER BY ...").fetchall()[0]
record_froid = ...
ville_chaude = ...
print(record_pluie)
print(record_froid)
print(ville_chaude)

In [ ]:
# Vérification
assert "record_pluie" in dir() and "record_froid" in dir() and "ville_chaude" in dir(), "Exécute d'abord ta cellule (elle a peut-être planté : lis la dernière ligne du message)."
assert record_pluie is not ... and record_froid is not ... and ville_chaude is not ..., "Remplace les ... par ton code."
assert record_pluie[2] == 22.1, f"Le record de pluie est 22.1 mm, tu as {record_pluie} : ORDER BY pluie_mm DESC."
assert record_froid[2] == 4.9, f"Le record de froid est 4.9 °C, tu as {record_froid} : ORDER BY temp_min (croissant)."
assert ville_chaude == "Nîmes", f"ville_chaude doit valoir Nîmes, tu as {ville_chaude}."
print("✅ OK")

## Partie 5 — Prévision, graphique et bulletin

### Étape 11 — La prévision naïve de chaque ville

**Objectif** : Réutiliser le modèle du module 11 : pour chaque ville, l'entraîner sur ses `temp_max` et prévoir le lendemain.

**À faire** :
1. La classe `ModeleMoyenne` est donnée.
2. Pour chaque ville de `tableau.index`, fabrique un modèle, entraîne-le (`fit`) sur la liste des `temp_max` de la ville, et range `predict(1)[0]` dans le dictionnaire `previsions`.

**Résultat attendu** :
```
{'Albi': 22.9, 'Montpellier': 25.2, 'Nîmes': 26.8, 'Perpignan': 25.3, 'Tarbes': 20.5, 'Toulouse': 24.0}
```

<details><summary>💡 Indice 1 : la méthode</summary>

Pour chaque ville : `fit` sur ses températures, puis `predict` pour 1 jour. `predict` renvoie une **liste** d'une valeur : prends sa position 0.

📖 Module 11, cours, section 9.

</details>

<details><summary>🔑 Indice 2 : la ligne à écrire (si l'indice 1 ne suffit pas)</summary>

`modele.fit(temperatures)` puis `previsions[ville] = modele.predict(1)[0]` (une liste d'une valeur : on prend la première).

</details>

In [ ]:
class ModeleMoyenne:
    def __init__(self):
        self.moyenne = None

    def fit(self, valeurs):
        self.moyenne = round(sum(valeurs) / len(valeurs), 1)

    def predict(self, nb_jours):
        previsions = []
        for jour in range(nb_jours):
            previsions.append(self.moyenne)
        return previsions


previsions = {}
for ville in tableau.index:
    temperatures = list(historique[historique["ville"] == ville]["temp_max"])
    modele = ModeleMoyenne()
    ...
    previsions[ville] = ...
print(previsions)

In [ ]:
# Vérification
assert "previsions" in dir(), "Exécute d'abord ta cellule."
assert len(previsions) == 6, f"previsions doit avoir 6 villes, il en a {len(previsions)}."
assert previsions.get("Toulouse") is not None, "La prévision de Toulouse vaut None : as-tu appelé modele.fit(temperatures) avant predict ?"
assert previsions == {'Albi': 22.9, 'Montpellier': 25.2, 'Nîmes': 26.8, 'Perpignan': 25.3, 'Tarbes': 20.5, 'Toulouse': 24.0}, f"previsions doit valoir {'Albi': 22.9, 'Montpellier': 25.2, 'Nîmes': 26.8, 'Perpignan': 25.3, 'Tarbes': 20.5, 'Toulouse': 24.0}, tu as {previsions}."
print("✅ OK")

### Étape 12 — Le graphique du bulletin

**Objectif** : Un graphique en barres de la température maximale moyenne par ville, dans l'ordre du tableau, titré et enregistré.

**À faire** :
1. Barres : villes de `tableau` en étiquettes, colonne `temp_max` en hauteurs.
2. Titre, légendes des deux axes.
3. Enregistre dans `sorties/bulletin_final.png`, avant `plt.show()`.

**Résultat attendu** :
```
Six barres, de la plus chaude à la moins chaude.
```

<details><summary>💡 Indice 1 : la méthode</summary>

Le graphique en barres du module 10 : étiquettes = les villes de `tableau` (son index), hauteurs = la colonne `temp_max` ; titre, axes, puis `savefig` **avant** `show`.

📖 Module 10, cours, section 10.

</details>

<details><summary>🔑 Indice 2 : la ligne à écrire (si l'indice 1 ne suffit pas)</summary>

`plt.bar(list(tableau.index), list(tableau["temp_max"]))`, puis `plt.title(...)`, `plt.xlabel(...)`, `plt.ylabel(...)`, `plt.savefig("sorties/bulletin_final.png")`, et enfin `plt.show()`.

</details>

In [ ]:
plt.bar(..., ...)
plt.title(...)
plt.xlabel(...)
plt.ylabel(...)
plt.savefig(...)
plt.show()

In [ ]:
# Vérification
assert Path("sorties/bulletin_final.png").exists(), "L'image sorties/bulletin_final.png n'existe pas : vérifie le chemin dans savefig."
assert Path("sorties/bulletin_final.png").stat().st_size > 5000, "L'image est presque vide : savefig doit venir AVANT plt.show()."
print("✅ OK")

### Étape 13 — Le bulletin texte

**Objectif** : Écrire le bulletin lisible par un humain, dans `sorties/bulletin_final.txt`.

**À faire** :
1. `lignes` commence par le titre et une ligne vide.
2. Pour chaque ville de `tableau`, une ligne comme : `Nîmes : max 26.8 °C, min 13.0 °C, pluie 123.7 mm (15 jours), demain 26.8 °C`.
3. Puis une ligne vide et trois lignes de records (voir le résultat attendu).
4. La cellule assemble `bulletin` (une ligne + `"\n"` à chaque tour) et l'écrit dans le fichier.

**Résultat attendu** :
```
BULLETIN MÉTÉOLOCALE — SEPTEMBRE 2026

Nîmes : max 26.8 °C, min 13.0 °C, pluie 123.7 mm (15 jours), demain 26.8 °C
Perpignan : max 25.3 °C, min 15.0 °C, pluie 135.5 mm (20 jours), demain 25.3 °C
Montpellier : max 25.2 °C, min 13.7 °C, pluie 112.2 mm (14 jours), demain 25.2 °C
Toulouse : max 24.0 °C, min 11.8 °C, pluie 107.8 mm (15 jours), demain 24.0 °C
Albi : max 22.9 °C, min 11.4 °C, pluie 100.7 mm (17 jours), demain 22.9 °C
Tarbes : max 20.5 °C, min 8.9 °C, pluie 60.1 mm (11 jours), demain 20.5 °C

Ville la plus chaude : Nîmes
Journée la plus pluvieuse : Nîmes le 2026-09-11 (22.1 mm)
Nuit la plus froide : Tarbes le 2026-09-29 (4.9 °C)
```

<details><summary>💡 Indice 1 : la méthode</summary>

Une f-string par ville, avec les variables déjà préparées et `previsions[ville]`. Pour les records : `record_pluie` est un tuple (ville, date, valeur) ; `record_froid` est organisé **pareil** : ville, date, valeur.

📖 Module 02 (f-strings) ; module 08 (écrire un fichier).

</details>

<details><summary>🔑 Indice 2 : la ligne à écrire (si l'indice 1 ne suffit pas)</summary>

Une f-string par ville avec les variables déjà préparées et `previsions[ville]`. Pour les records : `record_pluie[0]` (ville), `[1]` (date), `[2]` (valeur).

</details>

In [ ]:
lignes = ["BULLETIN MÉTÉOLOCALE — SEPTEMBRE 2026", ""]
for ville in tableau.index:
    t_max = tableau["temp_max"][ville]
    t_min = tableau["temp_min"][ville]
    pluie = tableau["pluie_totale"][ville]
    jours = tableau["jours_pluie"][ville]
    lignes.append(...)
lignes.append("")
lignes.append(f"Ville la plus chaude : {ville_chaude}")
lignes.append(...)
lignes.append(...)

bulletin = ""
for ligne in lignes:
    bulletin = bulletin + ligne + "\n"
with open("sorties/bulletin_final.txt", "w", encoding="utf-8") as fichier:
    fichier.write(bulletin)
print(bulletin)

In [ ]:
# Vérification
assert "lignes" in dir() and "bulletin" in dir(), "Exécute d'abord ta cellule."
attendues = ['BULLETIN MÉTÉOLOCALE — SEPTEMBRE 2026', '', 'Nîmes : max 26.8 °C, min 13.0 °C, pluie 123.7 mm (15 jours), demain 26.8 °C', 'Perpignan : max 25.3 °C, min 15.0 °C, pluie 135.5 mm (20 jours), demain 25.3 °C', 'Montpellier : max 25.2 °C, min 13.7 °C, pluie 112.2 mm (14 jours), demain 25.2 °C', 'Toulouse : max 24.0 °C, min 11.8 °C, pluie 107.8 mm (15 jours), demain 24.0 °C', 'Albi : max 22.9 °C, min 11.4 °C, pluie 100.7 mm (17 jours), demain 22.9 °C', 'Tarbes : max 20.5 °C, min 8.9 °C, pluie 60.1 mm (11 jours), demain 20.5 °C', '', 'Ville la plus chaude : Nîmes', 'Journée la plus pluvieuse : Nîmes le 2026-09-11 (22.1 mm)', 'Nuit la plus froide : Tarbes le 2026-09-29 (4.9 °C)']
assert len(lignes) == len(attendues), f"lignes doit contenir {len(attendues)} lignes, elle en contient {len(lignes)}."
for i in range(len(attendues)):
    assert lignes[i] == attendues[i], f"Ligne {i} attendue :\n  {attendues[i]}\nTu as :\n  {lignes[i]}"
assert Path("sorties/bulletin_final.txt").read_text(encoding="utf-8") == bulletin, "Le fichier sorties/bulletin_final.txt ne contient pas le bulletin."
print("✅ OK")

### Étape 14 — Le bulletin JSON

**Objectif** : Le site web de MétéoLocale veut les mêmes informations en **JSON** (module 08), dans `sorties/bulletin_final.json`.

**À faire** :
1. Nouveau : un **dictionnaire dans un dictionnaire**. Une valeur de dictionnaire peut être n'importe quoi, y compris un autre dictionnaire : `villes["Albi"]` sera la fiche d'Albi, et `villes["Albi"]["temp_max"]` une rubrique de cette fiche (on lit de gauche à droite, une clé après l'autre). Le JSON sait ranger cette structure telle quelle.
2. Remplis `villes` : pour chaque ville, un dictionnaire `{"temp_max": ..., "temp_min": ..., "pluie_totale": ..., "jours_pluie": ..., "prevision": ...}`.
3. Attention : `json` ne connaît que les types Python de base. Les nombres venant de pandas se convertissent avec `float(...)` et `int(...)` (pour `jours_pluie`).
4. La cellule assemble `resume` et l'écrit avec `json.dump(..., ensure_ascii=False, indent=2)` : `ensure_ascii=False` garde les accents lisibles, `indent=2` met une rubrique par ligne.

**Résultat attendu** :
```
{
  "mois": "2026-09",
  "nb_releves": 166,
  "ville_la_plus_chaude": "Nîmes",
  "villes": { ... }
}
```

<details><summary>💡 Indice 1 : la méthode</summary>

Même forme que la rubrique `"temp_max"` pour les autres rubriques. `jours_pluie` est un nombre entier : `int(...)` ; les autres sont à virgule : `float(...)`. Le dictionnaire dans un dictionnaire est expliqué dans l'énoncé.

📖 Module 08 (`json.dump`).

</details>

<details><summary>🔑 Indice 2 : la ligne à écrire (si l'indice 1 ne suffit pas)</summary>

Même forme que `"temp_max"` : `float(tableau["temp_min"][ville])`, … et `int(tableau["jours_pluie"][ville])`.

</details>

In [ ]:
villes = {}
for ville in tableau.index:
    villes[ville] = {
        "temp_max": float(tableau["temp_max"][ville]),
        "temp_min": ...,
        "pluie_totale": ...,
        "jours_pluie": ...,
        "prevision": previsions[ville],
    }

resume = {
    "mois": "2026-09",
    "nb_releves": len(historique),
    "ville_la_plus_chaude": ville_chaude,
    "villes": villes,
}
with open("sorties/bulletin_final.json", "w", encoding="utf-8") as fichier:
    json.dump(resume, fichier, ensure_ascii=False, indent=2)
print(Path("sorties/bulletin_final.json").read_text(encoding="utf-8")[:300])

In [ ]:
# Vérification
assert Path("sorties/bulletin_final.json").exists(), "Le fichier sorties/bulletin_final.json n'existe pas."
texte = Path("sorties/bulletin_final.json").read_text(encoding="utf-8")
assert texte.strip() != "", "Le fichier JSON est vide : la cellule a planté pendant json.dump. Si le message dit 'not JSON serializable', convertis avec float(...) ou int(...)."
relu = json.loads(texte)
assert relu["nb_releves"] == 166, f"nb_releves doit valoir 166, tu as {relu['nb_releves']}."
assert len(relu["villes"]) == 6, "Il faut les 6 villes dans le JSON."
assert relu["villes"]["Albi"]["jours_pluie"] == 17, f"Albi doit avoir jours_pluie = 17, tu as {relu['villes']['Albi']['jours_pluie']}."
assert relu["villes"]["Tarbes"]["temp_min"] == 8.9, f"Tarbes doit avoir temp_min = 8.9, tu as {relu['villes']['Tarbes']['temp_min']}."
assert "Nîmes" in texte, "Les accents doivent rester lisibles : ensure_ascii=False."
print("✅ OK")

### Étape 15 — Tout en un : la fonction `pipeline`

**Objectif** : Inès veut relancer tout le nettoyage et le rangement en **une ligne** quand le fichier d'octobre arrivera. Écris `pipeline(chemin_brut, chemin_base)` qui : charge le CSV brut, appelle `nettoyer_villes` et `convertir_temperatures`, enlève doublons et cases vides, recrée la base, y range les relevés, **ferme** la connexion, et renvoie le nombre de relevés rangés.

**À faire** :
1. Ferme d'abord la connexion des étapes précédentes : `connexion.close()`.
2. Écris la fonction en réutilisant tes fonctions des étapes 3 et 4 (pas de copier-coller de leur contenu).
3. ⚠️ Dans la fonction, travaille **uniquement** sur la variable locale `tableau`, fabriquée à partir du paramètre `chemin_brut` : remplace `propre` (qui contient septembre) par `tableau` dans la boucle d'insertion. La vérification essaie ta fonction sur un **autre** fichier, celui d'octobre.
4. Teste-la sur `../donnees/meteo_brut.csv` avec la base `sorties/pipeline_test.db` ; range le résultat dans `nb_ranges`.

**Résultat attendu** :
```
166
```

<details><summary>💡 Indice 1 : la méthode</summary>

La fonction enchaîne tes deux fonctions de nettoyage, les deux méthodes de l'étape 5, puis les gestes de base de l'étape 7. Tout se fait sur la variable locale `tableau`, jamais sur `propre`.

📖 Module 07 (fonctions) ; module 12, cours, section 4.

</details>

<details><summary>🔑 Indice 2 : la ligne à écrire (si l'indice 1 ne suffit pas)</summary>

```python
def pipeline(chemin_brut, chemin_base):
    tableau = pd.read_csv(chemin_brut, sep=";")
    tableau = nettoyer_villes(tableau)
    tableau = convertir_temperatures(tableau)
    tableau = tableau.drop_duplicates().dropna()
    base = sqlite3.connect(chemin_base)
    base.execute("DROP TABLE IF EXISTS releves")
    base.execute("CREATE TABLE releves (date TEXT, ville TEXT, temp_min REAL, temp_max REAL, pluie_mm REAL, vent_kmh INTEGER)")
    for f in tableau.to_dict("records"):
        base.execute("INSERT INTO releves VALUES (?, ?, ?, ?, ?, ?)",
                     (f["date"], f["ville"], f["temp_min"], f["temp_max"], f["pluie_mm"], f["vent_kmh"]))
    base.commit()
    base.close()
    return len(tableau)
```

</details>

In [ ]:
connexion.close()


def pipeline(chemin_brut, chemin_base):
    tableau = pd.read_csv(chemin_brut, sep=";")
    tableau = nettoyer_villes(tableau)
    tableau = ...
    tableau = ...
    base = sqlite3.connect(chemin_base)
    # Recrée la table releves à neuf, insère chaque relevé de tableau (pas de propre !), valide.
    ...
    base.close()
    return len(tableau)


nb_ranges = pipeline("../donnees/meteo_brut.csv", "sorties/pipeline_test.db")
print(nb_ranges)

In [ ]:
# Vérification
assert "pipeline" in dir() and "nb_ranges" in dir(), "Exécute d'abord ta cellule (elle a peut-être planté : lis la dernière ligne du message)."
assert nb_ranges == 166, f"Sur le fichier de septembre, pipeline doit renvoyer 166, tu as {nb_ranges}."
from pathlib import Path
lignes_octobre = [
    "date;ville;temp_min;temp_max;pluie_mm;vent_kmh",
    "2026-10-01;Toulouse;11.2;22,1;0.0;14",
    "2026-10-01;  albi ;10.0;21.4;1.2;9",
    "2026-10-01;  albi ;10.0;21.4;1.2;9",
    "2026-10-02;Toulouse;12.0;21.4;;22",
    "2026-10-02;Tarbes;8.5;19,8;3.4;N/A",
    "2026-10-02;Nîmes;13.1;24.0;0.0;31",
]
with open("sorties/meteo_brut_octobre.csv", "w", encoding="utf-8") as fichier:
    for ligne in lignes_octobre:
        fichier.write(ligne + "\n")
n_oct = pipeline("sorties/meteo_brut_octobre.csv", "sorties/pipeline_octobre.db")
v = sqlite3.connect("sorties/pipeline_octobre.db")
tables = [t[0] for t in v.execute("SELECT name FROM sqlite_master WHERE type = 'table'").fetchall()]
lignes_oct = v.execute("SELECT date, ville, temp_max FROM releves").fetchall() if "releves" in tables else []
v.close()
assert "releves" in tables, "La base d'essai n'a pas de table releves : la fonction doit faire DROP, CREATE, les INSERT et commit."
assert len(lignes_oct) != 166, "Sur le fichier d'octobre, ta fonction a rangé les 166 relevés de SEPTEMBRE : dans la boucle, remplace propre par tableau (la variable de la fonction)."
assert n_oct == 3, f"Sur le petit fichier d'octobre (6 lignes : 1 doublon, 1 pluie vide, 1 vent N/A), il doit rester 3 relevés ; ta fonction renvoie {n_oct}."
assert len(lignes_oct) == 3, f"La base d'octobre contient {len(lignes_oct)} relevés au lieu de 3 : as-tu fait base.commit() avant base.close() ?"
assert sorted(l[1] for l in lignes_oct) == ["Albi", "Nîmes", "Toulouse"], f"Les villes d'octobre doivent être nettoyées (Albi, Nîmes, Toulouse) : tu as {sorted(l[1] for l in lignes_oct)}."
assert ("2026-10-01", "Toulouse", 22.1) in lignes_oct, "La température de Toulouse le 2026-10-01 doit valoir 22.1 (virgule réparée)."
print("✅ OK")

## 🏁 Ce que tu as construit

Un **pipeline complet**, du fichier abîmé jusqu'au bulletin :

| Étape | Ce qui a été produit | Module |
|---|---|---|
| Chargement, diagnostic | `brut`, `diagnostic` | 06, 10 |
| Nettoyage | `nettoyer_villes`, `convertir_temperatures`, `propre` (166 lignes) | 07, 10 |
| Base de données | `sorties/meteolocale_final.db` | 12 |
| Statistiques, records | `tableau`, `record_pluie`, `record_froid` | 10, 12 |
| Prévision | `previsions` (`ModeleMoyenne`) | 11 |
| Graphique | `sorties/bulletin_final.png` | 10 |
| Bulletin | `sorties/bulletin_final.txt`, `sorties/bulletin_final.json` | 08 |
| Automatisation | `pipeline(...)` | 07, 12 |

Inès relit le bulletin, sourit, et l'envoie à la direction à 11 h 58.

**Et maintenant ?** Tu as le socle de tout le reste de la formation : charger, nettoyer, ranger, analyser, prévoir,
restituer. Les prochains modules remplaceront notre `ModeleMoyenne` par de vrais modèles (scikit-learn), le
fichier par des API et des bases plus grandes. Mais les gestes resteront ceux-là.

Retourne cocher la checklist du `README.md` : tu l'as bien mérité.